
Same as final_dataset and adds two columns from full_data:

- `birth_month`
- `birth_year`

They're needed to turn `earliest_arrest_date` (arrests through 2024) into "arrested before 18" during cleaning.

In [1]:
from pathlib import Path
import pandas as pd

# Works whether the notebook runs from the repo root or from the logistical_regression_analysis/ folder.
REPO = next(p for p in [Path('.'), Path('..')] if (p / 'Data and Resources' / 'final_data.csv').exists())
DATA = REPO / 'Data and Resources'

final = pd.read_csv(DATA / 'final_data.csv')
birth = pd.read_csv(DATA / 'full_data.csv', usecols=['respondent_id', 'birth_month', 'birth_year'])
print('final_data.csv:', final.shape)
print('birth columns from full_data.csv:', birth.shape)

final_data.csv: (8984, 28)
birth columns from full_data.csv: (8984, 3)


In [2]:
# Join on respondent_id. validate='one_to_one' stops with an error if either file has a duplicate ID.
new = final.merge(birth, on='respondent_id', how='left', validate='one_to_one')

assert len(new) == len(final) == 8984
assert new['respondent_id'].is_unique
assert new['birth_month'].between(1, 12).all()
assert new['birth_year'].between(1980, 1984).all()
assert new[final.columns].equals(final)

print(new.shape)
print(new['birth_year'].value_counts().sort_index().to_string())

(8984, 30)
birth_year
1980    1691
1981    1874
1982    1841
1983    1807
1984    1771


In [3]:
OUT = REPO / 'logistical_regression_analysis' / 'final_data_steven.csv'
new.to_csv(OUT, index=False)
print('Saved', OUT, new.shape)
print(list(new.columns))

Saved ../logistical_regression_analysis/final_data_steven.csv (8984, 30)
['respondent_id', 'timely_college', 'violent_victimization_18_24', 'earliest_arrest_date', 'total_number_of_incarcerations', 'r_threatened_to_be_hurt_at_school', 'r_in_a_fight_at_school', 'days_r_absent_from_school', 'teachers_good', 'discipline_is_fair_agree_disagree', 'percent_of_peers_who_smoke', 'percent_of_peers_who_get_drunk_1_times_a_month', 'percent_of_peers_belong_to_a_gang', 'percent_of_peers_who_have_had_sex', 'what_year_first_arrest_arrest_10', 'bullying_victim_before_age_12', 'r_victim_of_bullying_12_18_years', 'percent_chance_r_victim_of_violent_crime_by_next_year', 'sex', 'r_ever_live_through_hard_times', 'household_net_worth', 'household_size', 'biological_father_education', 'biological_mother_education', 'race_ethnicity', 'asvab_math_verbal_score_percent', 'mother_serve_prison_sentence', 'father_serve_prison_sentence', 'birth_month', 'birth_year']
